## Avance 2: Análisis de Trayectoria en la Fórmula 1
Integrantes: Andy Villarroel, Javier Alcaino

Pregunta de investigación: ¿El promedio de puntos por carrera (PPC) de los pilotos que debutaron en "equipos top" es significativamente mayor que el de aquellos que debutaron en "equipos chicos"?

Introducción
En la Fórmula 1, el talento del piloto interactúa directamente con el rendimiento del monoplaza. El objetivo de este análisis es determinar estadísticamente si iniciar la carrera profesional en un equipo con historial ganador ("Equipo Top") asegura o correlaciona con un mayor éxito a lo largo de toda la trayectoria del piloto (medido en Puntos por Carrera), en comparación con debutar en un equipo sin campeonatos mundiales ("Equipo Chico").

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
import statsmodels.api as sm

# Configuración estética
sns.set_theme(style='whitegrid', font_scale=1.1)

# 1. Carga de datos
races = pd.read_csv('races.csv', na_values='\\N')
drivers = pd.read_csv('drivers.csv', na_values='\\N')
constructors = pd.read_csv('constructors.csv', na_values='\\N')
results = pd.read_csv('results.csv', na_values='\\N')
constructor_std = pd.read_csv('constructor_standings.csv', na_values='\\N')

# 2. Limpieza y Creación de Variables Clave (Código Original del Grupo)
# Definir equipos Top (ganadores de WCC)
ultima_carrera_x_año = races.sort_values('date').groupby('year')['raceId'].last().reset_index(name='raceId')
ids_top = constructor_std.merge(ultima_carrera_x_año, on='raceId').query('position == 1')['constructorId'].unique()
constructors['tipo'] = np.where(constructors['constructorId'].isin(ids_top), 'Top', 'Chico')

# Equipo de debut por piloto
debut = results.sort_values('raceId').groupby('driverId').first().reset_index()[['driverId', 'constructorId']]
debut = debut.rename(columns={'constructorId': 'constructor_debut'})
debut = debut.merge(constructors[['constructorId', 'name', 'tipo']], left_on='constructor_debut', right_on='constructorId')
debut = debut.rename(columns={'name': 'equipo_debut', 'tipo': 'tipo_debut'})

# Cálculo de PPC (mínimo 10 carreras para evitar ruido estadístico)
puntos_por_carrera = results.groupby('driverId').agg(pts_totales=('points', 'sum'), n_carreras=('raceId', 'count')).reset_index()
puntos_por_carrera = puntos_por_carrera.query('n_carreras >= 10')
puntos_por_carrera['ppc'] = puntos_por_carrera['pts_totales'] / puntos_por_carrera['n_carreras']

# Unir todo el dataset analítico
df = puntos_por_carrera.merge(debut[['driverId', 'equipo_debut', 'tipo_debut']], on='driverId')
df = df.merge(drivers[['driverId', 'forename', 'surname']], on='driverId')
df['nombre'] = df['forename'] + ' ' + df['surname']

print(f"Dataset listo. Pilotos analizados: {len(df)}")
print(f"Debut en Top: {(df['tipo_debut'] == 'Top').sum()} | Debut en Chico: {(df['tipo_debut'] == 'Chico').sum()}")

### 1. Análisis Exploratorio de Datos (EDA)
Observaremos las estadísticas descriptivas y la distribución del PPC entre ambos grupos para identificar patrones preliminares.

In [ ]:
# Estadísticas descriptivas
display(df.groupby('tipo_debut')['ppc'].describe())

# Visualizaciones
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
color_dict = {'Top': '#E10600', 'Chico': '#3A86FF'}

# 1. Histograma: Distribución del PPC
sns.histplot(data=df, x='ppc', hue='tipo_debut', kde=True, palette=color_dict, ax=axes[0], alpha=0.5)
axes[0].set_title('Histograma: Distribución de Puntos por Carrera')
axes[0].set_xlabel('Puntos Por Carrera (PPC)')

# 2. Boxplot: Detección de variabilidad y outliers
sns.boxplot(data=df, x='tipo_debut', y='ppc', palette=color_dict, ax=axes[1])
axes[1].set_title('Boxplot: Rango y Outliers del PPC')
axes[1].set_xlabel('Tipo de Equipo de Debut')
axes[1].set_ylabel('PPC')

# 3. Dispersión: PPC vs N° de Carreras Totales
sns.scatterplot(data=df, x='n_carreras', y='ppc', hue='tipo_debut', palette=color_dict, alpha=0.7, ax=axes[2])
axes[2].set_title('Dispersión: Experiencia vs Puntos')
axes[2].set_xlabel('Total de Carreras Disputadas')
axes[2].set_ylabel('Puntos Por Carrera (PPC)')

plt.tight_layout()
plt.show()

### 2. Test de Hipótesis (Prueba T para muestras independientes)
Hipótesis:

H₀: $\mu_{Top} = \mu_{Chico}$ (No hay diferencias significativas en el PPC promedio según el equipo de debut).

H₁: $\mu_{Top} \neq \mu_{Chico}$ (El PPC es significativamente distinto según el equipo de debut).

Nota: Utilizamos una Prueba T (equivalente a un ANOVA de 2 grupos) dado que comparamos las medias de dos grupos independientes.

In [ ]:
grupo_top = df[df['tipo_debut'] == 'Top']['ppc']
grupo_chico = df[df['tipo_debut'] == 'Chico']['ppc']

# Aplicamos la prueba T de Student
t_stat, p_val_t = stats.ttest_ind(grupo_top, grupo_chico, equal_var=False)

print("--- RESULTADOS PRUEBA DE HIPÓTESIS ---")
print(f"Estadístico T: {t_stat:.4f}")
print(f"p-valor: {p_val_t:.4e}")

if p_val_t < 0.05:
    print("\nConclusión: Como el p-valor < 0.05, RECHAZAMOS H0.")
    print("Existe evidencia estadística fuerte para afirmar que debutar en un equipo Top marca una diferencia en el rendimiento (PPC).")
else:
    print("\nConclusión: No rechazamos H0.")

### 3. Modelo de Regresión Lineal Múltiple
Para elevar el nivel del análisis, crearemos un modelo para predecir el PPC. ¿El rendimiento se explica solo por el equipo de debut, o también influye la cantidad de carreras (experiencia / longevidad) del piloto?

In [ ]:
# Convertir variable categórica 'tipo_debut' a numérica (Top = 1, Chico = 0)
df['es_top'] = np.where(df['tipo_debut'] == 'Top', 1, 0)

# Variables Independientes (X) y Dependiente (y)
X = df[['es_top', 'n_carreras']]
X = sm.add_constant(X) # Agregar el intercepto
y = df['ppc']

# Ajustar Modelo OLS
modelo_mult = sm.OLS(y, X).fit()
print(modelo_mult.summary())

# Extracción de métricas clave para la interpretación
r2 = modelo_mult.rsquared
f_stat_mod = modelo_mult.fvalue
p_val_mod = modelo_mult.f_pvalue
coef_top = modelo_mult.params['es_top']

print("\n--- INTERPRETACIÓN DE RESULTADOS DEL MODELO ---")
print(f"1. R-cuadrado ({r2:.4f}): Las variables 'Debut en Top' y 'Experiencia' explican el {r2*100:.2f}% de la varianza en los Puntos por Carrera.")
print(f"2. Estadístico F y su p-valor ({p_val_mod:.4e}): El modelo en su conjunto es estadísticamente significativo (p < 0.05).")
print(f"3. Coeficiente 'es_top' ({coef_top:.4f}): Manteniendo la experiencia constante, debutar en un equipo Top aumenta el PPC esperado en {coef_top:.2f} puntos a lo largo de la carrera del piloto.")

### 4. Conclusiones y Análisis Crítico (Metodología CA5)
¿Qué encontré?: La estadística descriptiva y el test de hipótesis arrojan evidencia contundente: el PPC promedio de los pilotos que debutan en equipos Top (1.62 pts) supera ampliamente al de los equipos chicos (0.63 pts). El p-valor cercano a cero rechaza firmemente la hipótesis nula. Además, la regresión lineal múltiple confirma que, independientemente de la longevidad del piloto, el equipo de origen impacta positivamente el PPC.

¿Qué significa?: En la F1, debutar en una escudería élite condiciona fuertemente el éxito profesional de un piloto. Esto podría explicarse mediante un "efecto embudo": los equipos grandes tienen mejores procesos para cazar talentos generacionales, y a su vez, empezar en un equipo dominante genera un sesgo de supervivencia en la parrilla, garantizando asientos más competitivos en el futuro.

¿Qué limita este análisis? (CRÍTICO): El sistema de puntuación de la Fórmula 1 ha cambiado drásticamente en la historia (ej. victorias valían 9 pts antes, y 25 pts ahora). Esto significa que el PPC de un piloto de los 90s estará artificialmente subestimado comparado con un piloto de la era híbrida.

Recomendaciones: Se sugiere al lector interpretar el R-cuadrado (21.7%) comprendiendo que, aunque la relación es altamente significativa, existen múltiples factores ocultos que dictan los puntos ganados (como el nivel del coche temporada a temporada y las fallas de motor).

Trabajo futuro: Para el avance final, se recomienda estandarizar los puntos históricos (aplicar el sistema moderno de 25-18-15 pts a todas las carreras desde 1950) para limpiar el ruido normativo. Asimismo, valdría la pena segmentar si el piloto "Chico" logró eventualmente dar el salto a un equipo "Top" más adelante en su carrera.